Matrix factorization with regularization 

In [68]:
!pip install matplotlib
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [52]:
import numpy as np 
import pandas as pd 

#read in files 
movies = pd.read_csv('/Users/mauraschorr/Downloads/ml-latest-small/movies.csv')
ratings = pd.read_csv('/Users/mauraschorr/Downloads/ml-latest-small/ratings.csv')
tags = pd.read_csv('/Users/mauraschorr/Downloads/ml-latest-small/tags.csv')

#combine into a full dataset 
moviesRatings = ratings.merge(movies, on="movieId", how="left")
print(moviesRatings.isnull().sum()) #there are no null values in this 

#now want to add the tags bc they'll be used for kNN
print(tags.isnull().sum()) #no nulls in the tags dataset 

userMovieTags = tags.groupby(
    ["userId", "movieId"]
)["tag"].apply(
    lambda x: " ".join(x.unique())
).reset_index()

#merge tags into full dataset
moviesRatings = moviesRatings.merge(
    userMovieTags,
    on=["userId", "movieId"],
    how="left"
)

print(moviesRatings['tag'].isnull().sum()) #99201
print(moviesRatings.isnull().sum()) #all 99201 nulls are tags 
moviesRatings


userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64
99201
userId           0
movieId          0
rating           0
timestamp        0
title            0
genres           0
tag          99201
dtype: int64


,userId,movieId,rating,timestamp,title,genres,tag
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,NaN
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,NaN
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,NaN
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,NaN
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,NaN
...,...,...,...,...,...,...,...
100831,610,166534,4.0,1493848402,Split (2017),Drama|Horror|Thriller,NaN
100832,610,168248,5.0,1493850091,John Wick: Chapter Two (2017),Action|Crime|Thriller,Heroic Bloodshed
100833,610,168250,5.0,1494273047,Get Out (2017),Horror,NaN
100834,610,168252,5.0,1493846352,Logan (2017),Action|Sci-Fi,NaN


In [54]:
#want to note i used the implementation tutorial linked in the assignment 

#parameters 
#steps- max num of gradient descent iterations- more steps means more learning, too small(100)- undertrained, to large(10000+) slow, diminishing returns 
#alpha- learning rate- how large each update step is(higher-faster but may overshoot, lower- slower but stable)
#beta- regularization parameter(overfitting v generalization)- small(0.001-0.01)-weak regularization, too large(0.5+) can have underfitting 
#this uses L2 regularization 

def matrix_factorization(R, P, Q, K, steps=5000, alpha=0.002, beta=0.02): 
    Q=Q.T 
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])): 
                if R[i] [j] > 0:  #ignores all missing ratings- needed because matrix is sparse 
                    eij = R[i][j] - np.dot(P[i,:],Q[:,j]) #computes prediction error 
                    for k in range(K): 
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k]) 
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j]) 
        eR = np.dot(P,Q)
        e = 0 
        for i in range(len(R)): 
            for j in range(len(R[i])): 
                if R[i][j] > 0: 
                    e = e + pow(R[i][j] - np.dot(P[i,:],Q[:,j]), 2) #error calculation
        if e < 0.001:  #stops if error is very small- convergence checking 
            break 
    return P, Q.T


In [96]:
#creating different size matricies to use 
#filtering based on the users 
userCounts = ratings.groupby("userId").size()
userCounts.describe() #min- 20, max- 2698, 25%- 35, 75%- 168 
#going to look at the middle 2 quartiles- users with 35 to 168 ratings- left us with only 314 values so expanding
#looking at  
middleUsers = userCounts[userCounts[userCounts >= 35] & userCounts[userCounts <= 168]].index #314 values

#to test different size samples from this subset 
#user5000 = middleUsers.sample(5000)

#do the same but for movies not users 
movieCounts = ratings.groupby("movieId").size()
movieCounts.describe() #min- 1, max- 329, 25%- 1, 75%- 9 
#going to look at movies with 2 to 10 ratings 
middleMovies = movieCounts[movieCounts[movieCounts >= 35] & movieCounts[movieCounts <= 168]].index #712 values 
#to test different size samples from this subset 

#middleMovies
#movie5000 = np.random.choice(middleUsers, size=5000, replace=False)




Index([  3,   5,   7,   8,   9,  10,  11,  14,  15,  16,
       ...
       587, 588, 589, 591, 592, 593, 601, 602, 604, 609],
      dtype='int64', name='userId', length=314)


/var/folders/r9/4wytp03x2bsbtk10lxx12qdr0000gn/T/ipykernel_12498/1477435133.py:6: FutureWarning: Operation between non boolean Series with different indexes will no longer return a boolean result in a future version. Cast both Series to object type to maintain the prior behavior.
  middleUsers = userCounts[userCounts[userCounts >= 35] & userCounts[userCounts <= 168]].index
/var/folders/r9/4wytp03x2bsbtk10lxx12qdr0000gn/T/ipykernel_12498/1477435133.py:15: FutureWarning: Operation between non boolean Series with different indexes will no longer return a boolean result in a future version. Cast both Series to object type to maintain the prior behavior.
  middleMovies = movieCounts[movieCounts[movieCounts >= 35] & movieCounts[movieCounts <= 168]].index #712 values


In [21]:
#example to run model 
#R[i][j] - rating of user i for movie j 

#going to convert dataset into a pivot table so can run the algorithm on it 
#for matrix factorization only need movies and ratings, not tags 


R = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

R = R.to_numpy()

#testing different size matricies 

#trying for only users with more than 25 ratings 
userCounts = ratings.groupby("userId").size()


N = len(R)      #number of users
M = len(R[0])   #number of movies 
K = 2 #the number of hidden features used to represent users and movies
    #user taste can be explained by K hidden dimensions 

P = np.random.rand(N, K) 
Q = np.random.rand(M, K) 

nP, nQ = matrix_factorization(R, P, Q, K) 
nR = np.dot(nP, nQ.T)

print(nR)

[[4.97707341 2.98261177 5.04296352 1.00229128]
 [3.98232809 2.40255691 4.23535178 1.00051394]
 [1.0045463  0.98792079 5.82926522 4.97152189]
 [0.99796292 0.90417572 4.82771367 3.98404003]
 [1.18442936 1.01339801 4.98521273 3.99043069]]


Take 2

In [1]:
import os
import sys

REPO_ROOT = "/Users/mauraschorr/Downloads/recommender-system"
SRC_PATH = os.path.join(REPO_ROOT, "src")

print("Notebook folder:", os.getcwd())
print("Repo exists:", os.path.exists(REPO_ROOT))
print("Src exists:", os.path.exists(SRC_PATH))
print("Files in src:", os.listdir(SRC_PATH))

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("First sys.path entry:", sys.path[0])


Notebook folder: /Users/mauraschorr/Downloads/recommender-system/notebook
Repo exists: True
Src exists: True
Files in src: ['temp', 'svd_rec.py', '__pycache__', 'nn_rec.py', 'data_utils.py', 'random_rec.py']
First sys.path entry: /Users/mauraschorr/Downloads/recommender-system/src


In [2]:
from data_utils import load_ratings, build_folds
from surprise import Dataset, Reader
import pandas as pd 

ratings_path = "/Users/mauraschorr/Downloads/ml-latest-small/ratings.csv"

ratings = pd.read_csv(ratings_path)

#convert dataset to surprise format 
reader = Reader(rating_scale=(1, 5))

ratings_data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

#build folds consistent with other datasets 
folds = build_folds(ratings_data)



In [4]:
#kNN 
import time
from surprise import KNNBasic, accuracy

#defining knn model 
def run_knn_cv(folds, k=80, user_based=False, save_predictions=True):
    fold_results = []
    all_predictions = []

    for fold_num, (trainset, testset) in enumerate(folds, start=1):
        sim_options = {
            "name": "cosine",
            "user_based": user_based #it compares movies to movies not users to users 
        }

        model = KNNBasic(
            k=k,
            min_k=1,
            sim_options=sim_options,
            verbose=False
        )

        #track the time it runs on each fold 
        start_time = time.time()

        model.fit(trainset)
        predictions = model.test(testset)

        end_time = time.time()
        elapsed_time = end_time - start_time

        #stats for the run on each fold 
        rmse = accuracy.rmse(predictions, verbose=False)
        mae = accuracy.mae(predictions, verbose=False)


        fold_results.append({
            "fold": fold_num,
            "k": k,
            "RMSE": rmse,
            "MAE": mae,
            "Time": elapsed_time
        })

        for pred in predictions:
            all_predictions.append({
                "fold": fold_num,
                "userId": pred.uid,
                "movieId": pred.iid,
                "actual_rating": pred.r_ui,
                "predicted_rating": pred.est,
                "algorithm": "KNN",
                "k": k
            })

    results_df = pd.DataFrame(fold_results)
    predictions_df = pd.DataFrame(all_predictions)

    #use for final run of code 
    if save_predictions:
        predictions_df.to_csv(f"knn_predictions_k80.csv", index=False)

    avg_rmse = results_df["RMSE"].mean()
    avg_mae = results_df["MAE"].mean()
    avg_time = results_df["Time"].mean()

    return results_df, predictions_df, avg_rmse, avg_mae, avg_time

In [5]:
#testing different k values 

#k values i want to test 
k_values = [4, 8, 10, 20, 40, 80, 100, 120]

#holds the results from the run with each k value 
summary_results = []

for k in k_values:

    fold_results, predictions, avg_rmse, avg_mae, avg_time = run_knn_cv(
        folds=folds,
        k=k,
        user_based=False,
        save_predictions=False
    )

    #save information from the runs with each k 
    summary_results.append({
        "k": k,
        "Average RMSE": avg_rmse,
        "Average MAE": avg_mae,
        "Average Time(sec)": avg_time
    })

knn_summary = pd.DataFrame(summary_results)
knn_summary = knn_summary.sort_values("Average RMSE")

knn_summary
#going to use k=80 because increasing the k after that doesn't significantly improve the RMSE or MAE after that and the time is similar 

,k,Average RMSE,Average MAE,Average Time(sec)
7,120,0.958611,0.744324,4.000842
6,100,0.961171,0.746734,3.677746
5,80,0.964464,0.749778,3.528284
4,40,0.977242,0.760655,3.183538
3,20,0.996517,0.776532,2.892554
2,10,1.025992,0.800367,2.887361
1,8,1.040930,0.811983,2.931152
0,4,1.106812,0.861850,2.999832


In [6]:
print("save test)")

save test)
